# Functional Data Wrangling with Lambda Functions

Difficulty: Beginner | ~25 min | Requires basic Python (lists, dicts, functions)

Compose Python's three functional tools — `filter()`, `map()`, and `sorted()` — each
handed a short **lambda**, to turn a messy JSON product catalog into a clean, ranked
table in one pipeline.

## Step 0 — Warm up: lambda, map, filter, sorted on plain numbers(Optional)

Before touching the messy catalog, see the four tools in their simplest form —
plain numbers, no missing fields to defend against. Each tool is one call with a
short lambda.

In [1]:
# A lambda is an anonymous one-expression function; the value after the colon
# is returned implicitly.
double = lambda x: x * 2
print(double(5))

# map() applies a function to every item; list() materializes the lazy result.
numbers = [1, 2, 3, 4, 5]
doubled = list(map(lambda x: x * 2, numbers))
print(doubled)

# filter() keeps only the items where the lambda returns True.
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(evens)

# sorted() reorders; the key lambda picks what to sort by, and -x flips the
# order to descending.
ranked_numbers = sorted(numbers, key=lambda x: -x)
print(ranked_numbers)

10
[2, 4, 6, 8, 10]
[2, 4]
[5, 4, 3, 2, 1]


## Step 1 - Install Dependencies

In [ ]:
pip install tabulate==0.10.0

## Step 2 — Our data

A messy, real-world-shaped product catalog loaded from `data/product_catalog.json`:
37 products across 5 categories. 

In [2]:
# Manual open/load/close; Python's "with" statement automates this — covered in Lab 2.
import json

f = open("data/product_catalog.json", encoding="utf-8")
catalog = json.load(f)
f.close()

print("Loaded", len(catalog), "product records.")

Loaded 37 product records.


## Step 3 — Inspect the mess first

Before transforming anything, look at the actual types and missing values so you know
what your code must survive. 

In [3]:
# Print each product's price type and stock value. .get() with a default of
# "MISSING" lets the loop run even for records with no stock key at all.
for product in catalog:
    price = product.get("price", product.get("pricing", {}).get("base"))
    stock = product.get("stock", product.get("qty", "MISSING"))
    print(product["id"], "| price type:", type(price).__name__,
          "| price:", repr(price), "| stock:", repr(stock),
          "| status:", product.get("status", "-"))

101 | price type: float | price: 25.99 | stock: 40 | status: -
102 | price type: float | price: 89.5 | stock: 0 | status: -
103 | price type: str | price: '34.99' | stock: 15 | status: -
104 | price type: float | price: 59.99 | stock: 22 | status: -
105 | price type: float | price: 19.99 | stock: 8 | status: -
106 | price type: float | price: 109.0 | stock: 12 | status: -
107 | price type: float | price: 249.99 | stock: 5 | status: -
108 | price type: NoneType | price: None | stock: 10 | status: -
109 | price type: float | price: 199.99 | stock: 'MISSING' | status: unavailable
201 | price type: float | price: 14.99 | stock: 60 | status: -
202 | price type: str | price: '29.99' | stock: 18 | status: -
203 | price type: float | price: 22.5 | stock: 0 | status: -
204 | price type: float | price: 45.0 | stock: 7 | status: -
205 | price type: float | price: 39.99 | stock: 3 | status: -
206 | price type: float | price: 54.99 | stock: 9 | status: -
207 | price type: float | price: 32.0 | stoc

## Step 4 — Part 1: select with filter()

Remove discontinued and out-of-stock products in one `filter()` call. 

In [4]:
# PART 1 — keep only products you could actually sell today.
# The predicate checks three things: not discontinued, some stock (> 0, counting
# the "qty" fallback key), and a usable price (not None).
active = list(filter(
    lambda product: not product["discontinued"]
                    and product.get("stock", product.get("qty", 0)) > 0
                    and product.get("price", product.get("pricing", {}).get("base")) is not None,
    catalog,
))

print("Active, in-stock products:", len(active))
for product in active:
    print(" ", product["id"], product["name"])

Active, in-stock products: 21
  101 Wireless Mouse
  103 USB-C Hub
  104 Bluetooth Earbuds
  106 Portable SSD 1TB
  107 Gaming Monitor 27in
  201 Stainless Steel Water Bottle
  202 Non-Stick Frying Pan
  204 Ceramic Dinner Set
  206 Vacuum Sealer
  208 Coffee Grinder
  301 Running Shoes
  303 Wool Beanie
  304 Leather Belt
  307 Cotton T-Shirt
  401 Yoga Mat
  402 Adjustable Dumbbells
  405 Insulated Water Bottle
  407 Resistance Bands Set
  501 Bestselling Novel
  502 Cookbook: World Cuisines
  505 Self-Help Bestseller


## Step 5a — Part 2a: normalize currency to USD

In [5]:
# PART 2a — normalize the mixed currencies to one unit: USD.
# to_usd() extracts the price from either location and converts it to USD, so every
# product is in the same unit before the EUR conversion and discount apply.
usd_per_currency = {"USD": 1.0, "EUR": 1.08, "GBP": 1.27}
usd_to_eur = 0.92
discount_rate = 0.10


def to_usd(product):
    if product.get("price") is not None:
        return float(product["price"]) * usd_per_currency[product.get("currency", "USD")]
    nested = product.get("pricing", {})
    return float(nested["base"]) * usd_per_currency[nested.get("currency", "USD")]

## Step 5b — Part 2b: convert, discount, round with `map()`

In [6]:
# PART 2b — convert to EUR, apply the 10% clearance discount, round to 2 decimals.
# One map() with a lambda reshapes each active product into a clean dictionary.
priced = list(map(lambda product: {
    "name": product["name"],
    "category": product["category"],
    "price_eur": round(to_usd(product) * usd_to_eur, 2),
    "sale_eur": round(to_usd(product) * usd_to_eur * (1 - discount_rate), 2),
}, active))

for product in priced:
    print(product["category"], "-", product["name"], "-> EUR", product["price_eur"])

Electronics - Wireless Mouse -> EUR 23.91
Electronics - USB-C Hub -> EUR 32.19
Electronics - Bluetooth Earbuds -> EUR 59.61
Electronics - Portable SSD 1TB -> EUR 100.28
Electronics - Gaming Monitor 27in -> EUR 229.99
Home & Kitchen - Stainless Steel Water Bottle -> EUR 13.79
Home & Kitchen - Non-Stick Frying Pan -> EUR 27.59
Home & Kitchen - Ceramic Dinner Set -> EUR 41.4
Home & Kitchen - Vacuum Sealer -> EUR 50.59
Home & Kitchen - Coffee Grinder -> EUR 16.1
Apparel - Running Shoes -> EUR 59.79
Apparel - Wool Beanie -> EUR 11.95
Apparel - Leather Belt -> EUR 22.99
Apparel - Cotton T-Shirt -> EUR 9.19
Sports & Outdoors - Yoga Mat -> EUR 18.39
Sports & Outdoors - Adjustable Dumbbells -> EUR 119.59
Sports & Outdoors - Insulated Water Bottle -> EUR 16.55
Sports & Outdoors - Resistance Bands Set -> EUR 13.8
Books - Bestselling Novel -> EUR 11.95
Books - Cookbook: World Cuisines -> EUR 22.99
Books - Self-Help Bestseller -> EUR 15.63


## Step 6 — Part 3: order with sorted() and a tuple key

Sort by category (A–Z), then by price within each category, highest first. 

In [7]:
# PART 3 — order by category (A-Z), then by price within each category,
# highest first. A tuple key lets sorted() compare two values at once; the
# minus sign flips the price to descending without touching the category.
ranked = sorted(priced, key=lambda product: (product["category"], -product["price_eur"]))

for product in ranked:
    print(product["category"], "-", product["name"], "- EUR", product["price_eur"])

Apparel - Running Shoes - EUR 59.79
Apparel - Leather Belt - EUR 22.99
Apparel - Wool Beanie - EUR 11.95
Apparel - Cotton T-Shirt - EUR 9.19
Books - Cookbook: World Cuisines - EUR 22.99
Books - Self-Help Bestseller - EUR 15.63
Books - Bestselling Novel - EUR 11.95
Electronics - Gaming Monitor 27in - EUR 229.99
Electronics - Portable SSD 1TB - EUR 100.28
Electronics - Bluetooth Earbuds - EUR 59.61
Electronics - USB-C Hub - EUR 32.19
Electronics - Wireless Mouse - EUR 23.91
Home & Kitchen - Vacuum Sealer - EUR 50.59
Home & Kitchen - Ceramic Dinner Set - EUR 41.4
Home & Kitchen - Non-Stick Frying Pan - EUR 27.59
Home & Kitchen - Coffee Grinder - EUR 16.1
Home & Kitchen - Stainless Steel Water Bottle - EUR 13.79
Sports & Outdoors - Adjustable Dumbbells - EUR 119.59
Sports & Outdoors - Yoga Mat - EUR 18.39
Sports & Outdoors - Insulated Water Bottle - EUR 16.55
Sports & Outdoors - Resistance Bands Set - EUR 13.8


## Step 7 — The close cousin: list comprehensions(Optional)

Everything `map()` does can also be written as a **list comprehension**. Both build
the same list; knowing both lets you read (and write) either style in other people's
code.

In [8]:
# map() + lambda and a list comprehension build the same list; pick whichever
# reads better for the task at hand. This is the comprehension twin of Part 2.
priced_alt = [{
    "name": product["name"],
    "category": product["category"],
    "price_eur": round(to_usd(product) * usd_to_eur, 2),
    "sale_eur": round(to_usd(product) * usd_to_eur * (1 - discount_rate), 2),
} for product in active]

print("map() and comprehension agree:", priced_alt == priced)

map() and comprehension agree: True


## Step 8 — The payoff: one expression, three tools

Compose the whole job — filter the mess, reshape it, rank it — as one pipeline. 

In [9]:
# The whole job in one expression: filter the mess, reshape it, rank it.
# Read inside out — filter runs first, then map, then sorted.
final = sorted(
    map(
        lambda product: {
            "name": product["name"],
            "category": product["category"],
            "price_eur": round(to_usd(product) * usd_to_eur, 2),
            "sale_eur": round(to_usd(product) * usd_to_eur * (1 - discount_rate), 2),
        },
        filter(
            lambda product: not product["discontinued"]
                            and product.get("stock", product.get("qty", 0)) > 0
                            and product.get("price", product.get("pricing", {}).get("base")) is not None,
            catalog,
        ),
    ),
    key=lambda product: (product["category"], -product["price_eur"]),
)

print("Final ranked list:", len(final), "products")

Final ranked list: 21 products


## Step 9 — Present the result as a table

A list of dictionaries is correct but hard to scan. `tabulate` turns rows into
a clean text table with column headers — a much better way to read results.

In [10]:
# A list of dictionaries is correct but hard to scan. tabulate turns rows into
# a clean text table with column headers — a much better way to read results.
from tabulate import tabulate

rows = [(product["category"], product["name"], product["price_eur"], product["sale_eur"]) for product in final]
print(tabulate(rows, headers=["Category", "Name", "Price (EUR)", "Sale (EUR)"], tablefmt="grid"))

+-------------------+------------------------------+---------------+--------------+
| Category          | Name                         |   Price (EUR) |   Sale (EUR) |
+===================+==============================+===============+==============+
| Apparel           | Running Shoes                |         59.79 |        53.81 |
+-------------------+------------------------------+---------------+--------------+
| Apparel           | Leather Belt                 |         22.99 |        20.69 |
+-------------------+------------------------------+---------------+--------------+
| Apparel           | Wool Beanie                  |         11.95 |        10.76 |
+-------------------+------------------------------+---------------+--------------+
| Apparel           | Cotton T-Shirt               |          9.19 |         8.27 |
+-------------------+------------------------------+---------------+--------------+
| Books             | Cookbook: World Cuisines     |         22.99 |        

## Step 10 — When a lambda is the wrong tool(Optional)

Lambdas win for one-line predicates. But this predicate has three moving parts
(discontinued flag, missing stock across two keys, `null` price check) and is used
twice in this lab — enough that a named function reads better and can be reused
without re-typing the whole expression. Same result either way.

In [11]:
# Lambdas win for one-line predicates. But this predicate has three moving
# parts (discontinued flag, missing stock across two keys, null price) — enough
# that a named function reads better and can be reused without re-typing it.
def is_active(product):
    stock = product.get("stock", product.get("qty", 0))
    price = product.get("price", product.get("pricing", {}).get("base"))
    return not product["discontinued"] and stock > 0 and price is not None

via_function = list(filter(is_active, catalog))
via_lambda = list(filter(
    lambda product: not product["discontinued"]
                    and product.get("stock", product.get("qty", 0)) > 0
                    and product.get("price", product.get("pricing", {}).get("base")) is not None,
    catalog,
))

print("Same result:", via_function == via_lambda, "| kept:", len(via_function))

Same result: True | kept: 21


## Section 11 — Optional Exercise

Swap the target currency: price everything in **pounds sterling** instead of euros
(rate `usd_to_gbp = 0.79`). Rename `usd_to_eur` to `usd_to_gbp`, the `price_eur` and
`sale_eur` fields to `price_gbp` and `sale_gbp`, and update the table headers and
print statements accordingly. Because the same rate multiplies every price, the
Part 3 ranking order must stay identical — confirm that the order matches the table
in Section 9.

Swap the target currency: instead of euros, price everything in **pounds sterling** (rate `usd_to_gbp = 0.79`). Rename `usd_to_eur` to `usd_to_gbp`, the `price_eur` and `sale_eur` fields to `price_gbp` and `sale_gbp`, and update the table headers and print statements accordingly. Because the same rate multiplies every price, the Part 3 ranking order must stay identical — confirm that by comparing the order of the final table to the one in Section 5.